In [1]:
import os
import sys
from pathlib import Path

import torch
import pandas as pd

PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

from preprocess import DemuxManager
from classifier import HTClassifier
from evaluator import evaluate_demux
manager = DemuxManager()

Project root: /home/chongxiao/projects/HT_Demux


In [ ]:

DATA_DIR = PROJECT_ROOT / ""

mtx_path  = DATA_DIR / "matrix.mtx.gz"
feat_path = DATA_DIR / "features.tsv.gz"
bar_path  = DATA_DIR / "barcodes.tsv.gz"

assert mtx_path.exists(), f"Missing {mtx_path}"
assert feat_path.exists(), f"Missing {feat_path}"
assert bar_path.exists(), f"Missing {bar_path}"

print("Data directory:", DATA_DIR)


Data directory: /home/chongxiao/projects/HT_Demux/C3


In [48]:
print("Loading data...")
counts, barcodes = manager.load_10x_mtx(
    mtx_path,
    feat_path,
    bar_path
)

manager.build_configs(max_klet=2)

print("Counts shape:", counts.shape)
print("Number of barcodes:", len(barcodes))


Loading data...
Counts shape: torch.Size([188267, 128])
Number of barcodes: 188267


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Running optimizer on {device}...")
results = manager.run_demux(
    counts,
    model_type="GMM",     # or "NB"
    method="GD",          # or "EM"
    device=device
)

In [45]:
clf = HTClassifier(
    cfg_labels=results["cfg_labels"],
    tag_names=results["tag_names"],
    configs=manager.configs
)

df_results = clf.classify(
    posteriors=results["posteriors"],
    barcodes=barcodes,
    map_thresh=0.1
)

df_results.head()

,barcode,assignment_raw,assignment,assignment_final,probability,p_HTO_sim_01,p_HTO_sim_02,p_HTO_sim_03,p_HTO_sim_04,p_HTO_sim_05,...,p_HTO_sim_87,p_HTO_sim_88,p_HTO_sim_89,p_HTO_sim_90,p_HTO_sim_91,p_HTO_sim_92,p_HTO_sim_93,p_HTO_sim_94,p_HTO_sim_95,p_HTO_sim_96
0,droplet_00000,HTO_sim_01+HTO_sim_33,Multiplet,Unassigned,0.082770,0.998166,0.047879,0.000050,0.000873,0.001118,...,0.000103,0.000999,0.006295,0.004879,0.011411,0.006273,0.044114,0.001070,0.059143,0.019483
1,droplet_00001,HTO_sim_78+HTO_sim_82,Multiplet,Unassigned,0.081105,0.000352,0.017924,0.000086,0.001575,0.000387,...,0.000123,0.003847,0.007878,0.012365,0.009739,0.011052,0.048236,0.002258,0.009274,0.013095
2,droplet_00002,HTO_sim_04+HTO_sim_85,Multiplet,Multiplet,0.898064,0.000161,0.004014,0.000079,0.987123,0.000031,...,0.000016,0.000152,0.000882,0.001030,0.001188,0.001274,0.005246,0.000104,0.002743,0.004018
3,droplet_00003,HTO_sim_05+HTO_sim_06,Multiplet,Multiplet,0.849255,0.000106,0.003932,0.000051,0.000193,0.999926,...,0.000081,0.000398,0.000843,0.000605,0.001555,0.001375,0.002922,0.000087,0.001406,0.001586
4,droplet_00004,HTO_sim_14+HTO_sim_70,Multiplet,Unassigned,0.033194,0.000225,0.024711,0.004111,0.001006,0.000332,...,0.002781,0.000921,0.007912,0.014380,0.012022,0.011433,0.026453,0.000347,0.023605,0.027233


In [46]:

OUT_DIR = PROJECT_ROOT / "results"
OUT_DIR.mkdir(exist_ok=True)

pred_path = OUT_DIR / "demux_results.csv"
df_results.to_csv(pred_path, index=False)

print(f"Demux complete. Results saved to {pred_path}")


Demux complete. Results saved to /home/chongxiao/projects/HT_Demux/results/demux_results.csv


In [ ]:
truth_path = DATA_DIR / "ground_truth_droplet.csv"
assert truth_path.exists(), f"Missing {truth_path}"

evaluate_demux(
    pred_path=pred_path,
    truth_path=truth_path
)